# references

Writes a Table reference for each table your Project reads, from its DESCRIBE.

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Key**: the columns whose values name exactly one row of a table.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.
- **Reference sheet**: references/reference_sheet.csv, one table listing every column of every table the Project has a Table reference for.
- **Database module**: a file such as references/ops.py, which gives each table of one database by name, such as ops.jobs.

## Before you start

Fill in the names cell below and save the notebook: each call reads it, and so does `project.check()`, which names the next step.

## Decision rules

- If you want the steps run for you, as most do: fill in the names cell and save the notebook, run "Run up to your Decisions", paste what it prints into the decisions cell under it, fill it in and save, then run "Run the rest". Each runs every step left in turn, up to the first only you can do, and says where it stopped.
- If you run the steps yourself, run each step's cell under "Step by step", in turn, in order.
- If you want only the steps that change files run for you, run `project.do_next()`, the last cell. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each table in `TABLES` has its Table reference in references/, and leaves no Key open.
- references/reference_sheet.csv lists each table's columns.
- `project.check()` names no next step of this Workflow.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
TABLES = ["ops.job_runs"]  # the tables, by their names in the warehouse, such as ["mart.daily_job_runs"]

## Run up to your Decisions

This cell and "Run the rest" are one call, `project.run_workflow`, run once before you answer and once after. Run the cell below: it runs every step left, up to the first only you can do, and says where it stopped. At a Decision, it prints a whole DECISIONS: paste it into the decisions cell under it, fill in each <FILL IN>, and save the notebook.

In [ ]:
# Run up to your Decisions: every step left, up to the first only you can do, such as a
# Decision, whose DECISIONS it prints to paste into the cell under it.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("references", query=query, execute=execute)

In [ ]:
DECISIONS = {"ops.job_runs": {"key": ["run_id"]}}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"key": ["job_id"], "types": {"fails": "bigint"},
#                          "tags": {"PERIOD_END": "'2026-10-31'"}}}
# "key": None says no columns name one row. project.check() names each one open, and
# prints a whole DECISIONS to paste over this line: fill in each <FILL IN>. A Key
# convert or new_table proposed, marked # proposed, is kept unless you answer "key"
# here; one add_references proposed stays open until you accept it.
# "key": "proposed" accepts either as if typed. "all_tables" answers for every table
# of this notebook, and a table's own answer overwrites it, kind by kind and tag by
# tag:
# {"all_tables": {"key": "proposed",
#                 "tags": {"PERIOD_END": "'2026-10-31'"}}}
# accepts each table's proposed Key and gives each table holding PERIOD_END that
# value. A table's location is placed under "location_root" in
# admin/databases.py; answer "location" only for one it can't place.
# A tag may be answered with a Run value's function of your Project's, imported above
# DECISIONS, such as "PERIOD_END": period_end. For a table's latest Partition, write
# project.new_run_value("period_end", latest_partition="ops.job_runs.dt") at the top
# of this cell, the import it prints under it, and run the cell once: it writes the
# function into building_blocks/run_values.py. And "seed", such as
# {"dt": {"from": "2026-01-01"}}, limits the Partitions a table that reads itself
# copies: how-to 09 shows both.

## Run the rest

Run the cell below once the decisions cell above is filled in and saved: the same call again, it goes on from where the cell before stopped, and it too stops at the first step only you can do: a refusal, a compare that failed, or a Decision left open, each named. Mend what it names, such as an answer in the decisions cell, save, and run this cell again.

In [ ]:
# Run the rest: the same call, run again once you have answered: it goes
# on from where it stopped, and stops again at a refusal, a compare that
# failed or a Decision left open.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("references", query=query, execute=execute)

# Step by step

Each step's own cell, to run or run again one step on purpose; the cells above run them all for you.

## 1. Write the Table references from DESCRIBE

It is a call to the warehouse, yours to run.

- **Goal:** a Table reference for each table in `TABLES`, its columns and their types read from its DESCRIBE, and its Key proposed and checked.
- **Action:** Run the cell below. It reads each table's DESCRIBE, counts the distinct values of each id column on its latest Partition, and checks the Key it proposes with `check_key`, all through your query, and writes only files.
- **Done when** each table in `TABLES` has its Table reference in references/; then step 2.

In [ ]:
project.add_references(TABLES, query=query)

## 2. Answer each table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each table's open Key: the Key step 1 proposed, accepted, or the columns whose values name exactly one row, or None when no columns do.
- **Action:** Run the cell below, `project.check()`: it names each open Decision and prints a whole `DECISIONS` to paste over the decisions cell's; then fill in each `<FILL IN>` in it, if any, and save the notebook. Step 1 proposed a Key for each table it could, from its columns and a count of each id column's distinct values, and checked it with `check_key`, printing each verdict. A proposed Key is pasted as it is, which accepts it; for two tables or more, `"all_tables": {"key": "proposed"}` accepts each. If no Key was proposed for a table, or its verdict says it repeats, then write its Key in place of its `<FILL IN>`, as a list such as `["run_id"]`, or None when no columns name one row; how-to 21 shows how to check one with `check_key` first.
- **Done when** the decisions cell answers each Key `project.check()` names for these tables; then step 3.

In [ ]:
project.check()

## 3. Write each answer into its table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its table's Table reference.
- **Action:** Run the cell below. Step 1 proposed each Table reference's first line, what one row is, from its Key, and its `does_not_add_up=`, the columns whose names say their values can't be added up across rows, such as an average, each marked `# proposed`; writing a Key proposes its first line again. Read them, and change any that is wrong, such as a distinct count no name gives away. The cell names each TODO line left, as a table with no Key keeps its first line's; no step waits on them, so fill each in when you like.
- **Done when** each table's Table reference leaves no Key open; then step 4.

In [ ]:
project.write_decisions(DECISIONS, tables=TABLES)

## 4. Refresh the Reference sheet and the Database modules

It changes only files, so project.do_next() runs it for you.

- **Goal:** each table in the Reference sheet and in its Database module.
- **Action:** Run the cell below. If you change a Table reference after it, such as filling in a TODO line, then run it again: `project.check()` names this step again, saying which Table reference changed.
- **Done when** references/reference_sheet.csv is written again after each table's Table reference last changed; the Workflow is done: go through the Self-check at the top.

In [ ]:
project.refresh_reference_sheet(TABLES)

In [ ]:
project.do_next()